# Практика 6. Основные структуры pandas: Series и DataFrame

На практике 2 мы считали отчёт вручную: цикл, условие, словарь-накопитель. Сегодня тот же набор
операций мы делаем инструментом, который для этого создан, — и на порядок быстрее.

**Данные пары:** журнал грузовых рейсов транспортной компании за март 2025 — 400 рейсов,
маршруты, перевозчики, вес, расстояние, стоимость, задержки.

**Из чего состоит пара:**

| Часть | О чём |
|---|---|
| 1 | `Series` — одномерная структура: создание и исследование |
| 2 | `DataFrame` — таблица: создание и первичное исследование |
| 3 | Индексация: `loc` и `iloc` |
| 4 | Фильтрация и выбор данных |
| 5 | Изменение структуры и вычисляемые признаки |
| 6 | Применение функций к строкам и столбцам |
| 7 | Мини-проект: карточка маршрута |

> Ячейки запускаются **Shift + Enter**, строго по порядку: каждая следующая использует
> переменные из предыдущих.

---
## Шаг 0. Проект (5 минут, повторение)

```
# терминал
mkdir pandas-trips
cd pandas-trips
```

`requirements.txt`:

```
pandas
```

```
# терминал (Windows)
python -m venv .venv
.\.venv\Scripts\Activate.ps1
pip install -r requirements.txt
```

```
# терминал (macOS / Linux)
python3 -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
```

`.gitignore` с `.venv/`, `git init`, ноутбук в папку проекта, kernel — `.venv`.

---
# Часть 1. Series

**`Series`** — одномерный набор значений **с индексом**. Это не просто список: у каждого значения
есть метка, по которой его можно достать, и все значения одного типа.

Проще всего думать так: `Series` — это один столбец таблицы, а `DataFrame` — таблица целиком.

Главное свойство, из-за которого pandas вообще нужен: **операции применяются сразу ко всем
элементам**, без цикла. Это называется векторизацией.

In [ ]:
import pandas as pd

# Из списка: индекс создастся автоматически — 0, 1, 2, ...
weights = pd.Series([12.5, 8.0, 21.3, 15.1])
print(weights)
print()

# Из словаря: ключи становятся индексом
tariffs = pd.Series({
    "Контейнер": 7.5,
    "Нефтепродукты": 9.2,
    "Зерно": 5.1,
    "Металл": 6.8,
})
print(tariffs)
print()

print("индекс:  ", list(tariffs.index))
print("значения:", list(tariffs.values))
print("тип значений:", tariffs.dtype)
print("размер:", tariffs.size)

In [ ]:
# Доступ: по метке и по позиции
print("по метке 'Зерно':      ", tariffs["Зерно"])
print("по позиции (iloc[2]):  ", tariffs.iloc[2])
print()

# Векторизация: операция применяется ко всем элементам сразу, цикл не нужен
print("тарифы +10%:")
print((tariffs * 1.1).round(2))
print()

# Встроенная статистика
print("среднее:", tariffs.mean().round(2))
print("максимум:", tariffs.max(), "— это", tariffs.idxmax())
print()
print(tariffs.describe().round(2))

### Задание 1

Дан `Series` с весом груза по рейсам (индекс — номер рейса).

1. Выведите вес рейса `T003` по метке.
2. Выведите вес второго рейса по позиции.
3. Посчитайте суммарный и средний вес.
4. Создайте новый `Series` — стоимость перевозки, если тариф 6.8 руб. за тонно-километр,
   а расстояние для всех рейсов 500 км. Округлите до целых.

In [ ]:
cargo = pd.Series([12.5, 8.0, 21.3, 15.1, 4.2],
                  index=["T001", "T002", "T003", "T004", "T005"])
print(cargo)

# 1) вес рейса T003 по метке
print("T003:", ...)

# 2) вес второго рейса по позиции (помните: позиции считаются с нуля)
print("второй рейс:", ...)

# 3) суммарный и средний вес
print("сумма:", ...)
print("среднее:", ...)

# 4) стоимость перевозки: вес * 500 км * 6.8 руб.
cost = ...
print(cost)

---
# Часть 2. DataFrame

**`DataFrame`** — двумерная таблица: строки с индексом и именованные столбцы.
Каждый столбец — это `Series`, и у всех столбцов общий индекс.

Создать его можно из словаря (ключ — имя столбца), из списка словарей (каждый словарь — строка)
или прочитать из файла.

In [ ]:
# Способ 1: словарь списков — ключи становятся названиями столбцов
df_small = pd.DataFrame({
    "trip_id": ["T001", "T002", "T003"],
    "route": ["Москва-Казань", "Москва-Воронеж", "Казань-Самара"],
    "weight_t": [12.5, 8.0, 21.3],
})
print(df_small)
print()

# Способ 2: список словарей — каждый словарь это строка
df_small2 = pd.DataFrame([
    {"trip_id": "T001", "weight_t": 12.5},
    {"trip_id": "T002", "weight_t": 8.0},
])
print(df_small2)
print()

print("размер (строки, столбцы):", df_small.shape)
print("столбцы:", list(df_small.columns))

## Данные пары

Запустите ячейку ниже — она создаст `data/trips.csv`.

| Колонка | Смысл |
|---|---|
| `trip_id` | номер рейса |
| `date` | дата рейса |
| `route` | маршрут |
| `carrier` | перевозчик |
| `cargo_type` | тип груза |
| `weight_t` | вес груза, тонн (местами не заполнен) |
| `distance_km` | расстояние, км |
| `duration_h` | время в пути, часов |
| `cost_rub` | стоимость перевозки, руб. |
| `delay_h` | задержка, часов (0 — рейс по расписанию) |

In [ ]:
# ЯЧЕЙКА-ГЕНЕРАТОР ДАННЫХ. Запустите её (Shift+Enter), менять ничего не нужно.
# Создаёт data/trips.csv — журнал грузовых рейсов транспортной компании за март 2025.

import csv
import random
from pathlib import Path

random.seed(6)

routes = {
    "Москва-Казань": 820, "Москва-Санкт-Петербург": 710, "Москва-Нижний Новгород": 420,
    "Москва-Воронеж": 520, "Казань-Самара": 360, "Екатеринбург-Челябинск": 220,
    "Новосибирск-Омск": 680, "Санкт-Петербург-Петрозаводск": 430,
}
carriers = ["ТрансЛогистик", "РусКарго", "СибПуть", "ВолгаТранс", "АвтоДорСервис"]
# тариф в рублях за тонно-километр
tariffs = {
    "Контейнер": 7.5, "Нефтепродукты": 9.2, "Зерно": 5.1,
    "Металл": 6.8, "Лес": 4.6, "Стройматериалы": 5.9,
}

rows = []
for i in range(1, 401):
    route, distance = random.choice(list(routes.items()))
    cargo = random.choice(list(tariffs))
    carrier = random.choice(carriers)

    weight = round(random.uniform(3, 22), 1)
    speed = random.uniform(42, 68)
    delay = 0 if random.random() < 0.7 else random.choice([1, 2, 3, 4, 6, 8, 12])
    duration = round(distance / speed + delay, 1)
    cost = round(distance * weight * tariffs[cargo] * random.uniform(0.9, 1.15), -1)

    rows.append({
        "trip_id": f"T{i:04d}",
        "date": f"2025-03-{random.randint(1, 31):02d}",
        "route": route,
        "carrier": carrier,
        "cargo_type": cargo,
        # в 3% случаев вес не записали — пригодится, когда будем смотреть пропуски
        "weight_t": "" if random.random() < 0.03 else weight,
        "distance_km": distance,
        "duration_h": duration,
        "cost_rub": int(cost),
        "delay_h": delay,
    })

Path("data").mkdir(exist_ok=True)
with open("data/trips.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)

print("Готово. Создан файл data/trips.csv")
print("Строк:", len(rows))
print("Колонки:", list(rows[0].keys()))

## Первичное исследование таблицы

Четыре команды, с которых начинается работа с **любой** незнакомой таблицей:

| Команда | Что показывает |
|---|---|
| `df.shape` | сколько строк и столбцов |
| `df.head()` | первые 5 строк — глазами посмотреть на данные |
| `df.info()` | типы столбцов и количество заполненных значений |
| `df.describe()` | статистику по числовым столбцам |

Отдельно — `df["колонка"].value_counts()` для категорий и `df.isna().sum()` для пропусков.

In [ ]:
df = pd.read_csv("data/trips.csv", parse_dates=["date"])

print("размер:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
# Статистика по числовым столбцам
df.describe().round(1)

In [ ]:
# Категории: сколько рейсов по каждому типу груза
print(df["cargo_type"].value_counts())
print()

# Пропуски по всем столбцам
print("Пропуски:")
print(df.isna().sum())
print()

# Уникальные значения
print("маршрутов:", df["route"].nunique())
print("перевозчиков:", df["carrier"].nunique())
print(df["carrier"].unique())

### Задание 2

Ответьте на вопросы о датасете, **каждый — одной командой**. Числа запишите в ответах.

1. Сколько в таблице строк и столбцов?
2. Какой тип груза встречается чаще всего и сколько раз?
3. В каком столбце есть пропуски и сколько их?
4. Какова максимальная стоимость рейса?
5. Какая средняя задержка по всем рейсам?

In [ ]:
# 1)
print("строк и столбцов:", ...)

# 2)
print(...)

# 3)
print(...)

# 4)
print("максимальная стоимость:", ...)

# 5)
print("средняя задержка:", ...)

---
# Часть 3. Индексация: `loc` и `iloc`

**Выбор столбца:**

| Запись | Что вернёт |
|---|---|
| `df["cost_rub"]` | `Series` — один столбец |
| `df[["cost_rub"]]` | `DataFrame` из одного столбца |
| `df[["route", "cost_rub"]]` | `DataFrame` из двух столбцов |

Разница между одинарными и двойными скобками важна: от неё зависит, что вы получите на выходе
и какие методы к этому применимы.

**Выбор строк — два разных инструмента:**

| Инструмент | Работает по | Правая граница срезa |
|---|---|---|
| `df.loc[...]` | **меткам** индекса | **включается** |
| `df.iloc[...]` | **позициям** (номерам с нуля) | не включается |

Разное поведение границы — источник бесконечных ошибок. `df.loc[0:4]` вернёт 5 строк,
`df.iloc[0:4]` — четыре.

In [ ]:
# Столбцы
print(type(df["cost_rub"]), "<- одинарные скобки")
print(type(df[["cost_rub"]]), "<- двойные скобки")
print()
df[["trip_id", "route", "cost_rub"]].head(3)

In [ ]:
# iloc — по позициям. Правая граница НЕ включается
print("iloc[0:4] вернул строк:", len(df.iloc[0:4]))
print("loc[0:4] вернул строк: ", len(df.loc[0:4]))
print()

# iloc: [строки, столбцы] по номерам
print(df.iloc[0:3, 0:4])
print()

# loc: [строки, столбцы] по меткам
print(df.loc[0:2, ["trip_id", "route", "cost_rub"]])

In [ ]:
# Осмысленный индекс: сделаем номер рейса индексом — тогда loc становится удобным
df_idx = df.set_index("trip_id")
print(df_idx.loc["T0100"])
print()
print("стоимость рейса T0100:", df_idx.loc["T0100", "cost_rub"])
print()

# Вернуть индекс в обычный столбец
print(df_idx.reset_index().head(2))

### Задание 3

1. Выведите столбцы `trip_id`, `carrier`, `cost_rub` для **первых 7 строк**.
2. Выведите строки с позициями 10–14 (пять строк) — все столбцы.
3. Сделайте `trip_id` индексом и выведите всю информацию о рейсе `T0250`.
4. Получите расстояние рейса `T0250` одним обращением.

In [ ]:
# 1)
...

# 2)
...

# 3)
trips = df.set_index("trip_id")
...

# 4)
print("расстояние T0250:", ...)

---
# Часть 4. Фильтрация

Фильтр в pandas — это **маска**: столбец из `True`/`False`, который подставляется в скобки.
Сначала посмотрите на саму маску, это снимает половину вопросов.

**Три правила, которые нужно запомнить сразу:**

1. Условия соединяются `&` (и), `|` (или), `~` (не) — **не** `and`, `or`, `not`.
2. Каждое условие **обязательно** в круглых скобках: `(df["a"] > 1) & (df["b"] < 2)`.
3. Если результат фильтра вы собираетесь изменять — ставьте `.copy()`, иначе pandas предупредит,
   что вы правите вид на данные, а не сами данные.

Полезные методы для условий: `.isin([...])`, `.between(a, b)`, `.isna()`, `.str.contains("...")`.

In [ ]:
# Маска — это просто столбец из True/False
mask = df["distance_km"] > 700
print(mask.head())
print("подходит рейсов:", mask.sum())
print()

# Подставляем маску в скобки
long_trips = df[df["distance_km"] > 700]
print("длинные рейсы:", long_trips.shape)
print()

# Два условия: скобки обязательны, соединяем через &
delayed_long = df[(df["distance_km"] > 700) & (df["delay_h"] > 0)]
print("длинные И с задержкой:", len(delayed_long))

# Или — через |
cheap_or_short = df[(df["cost_rub"] < 20000) | (df["distance_km"] < 300)]
print("дешёвые ИЛИ короткие:", len(cheap_or_short))

In [ ]:
# isin — значение входит в список
fuel_metal = df[df["cargo_type"].isin(["Нефтепродукты", "Металл"])]
print("нефтепродукты и металл:", len(fuel_metal))

# between — диапазон, включая границы
mid = df[df["distance_km"].between(400, 600)]
print("расстояние 400-600 км:", len(mid))

# isna — пропуски
print("без указанного веса:", len(df[df["weight_t"].isna()]))

# str.contains — поиск по тексту
moscow = df[df["route"].str.contains("Москва")]
print("маршруты через Москву:", len(moscow))

# ~ — отрицание
not_moscow = df[~df["route"].str.contains("Москва")]
print("маршруты без Москвы:", len(not_moscow))

### Задание 4

Посчитайте, сколько рейсов удовлетворяет каждому условию:

1. Задержка 4 часа и больше.
2. Перевозчик `РусКарго` **и** стоимость больше 50 000 руб.
3. Груз `Зерно` или `Лес`, расстояние меньше 500 км.
4. Вес груза не указан (пропуск).
5. Маршруты, в названии которых есть `Санкт-Петербург`, у которых не было задержки.

In [ ]:
# 1)
print("задержка >= 4 ч:", ...)

# 2)
print("РусКарго дороже 50000:", ...)

# 3)
print("зерно/лес короче 500 км:", ...)

# 4)
print("вес не указан:", ...)

# 5)
print("Санкт-Петербург без задержки:", ...)

---
# Часть 5. Изменение структуры и вычисляемые признаки

**Вычисляемый признак** — новый столбец, посчитанный из существующих. Это основная работа
аналитика: сырые данные почти никогда не содержат нужного показателя напрямую.

| Операция | Как |
|---|---|
| новый столбец | `df["new"] = df["a"] / df["b"]` |
| удалить столбцы | `df.drop(columns=["a", "b"])` |
| переименовать | `df.rename(columns={"old": "new"})` |
| сменить тип | `df["a"].astype(int)` |
| отсортировать | `df.sort_values("cost_rub", ascending=False)` |
| разбить на категории | `pd.cut(df["a"], bins=[...], labels=[...])` |

Почти все методы **возвращают новую таблицу**, а не меняют старую. Поэтому результат нужно
присваивать: `df = df.rename(...)`. Параметр `inplace=True` существует, но пользоваться им
не стоит — он мешает строить цепочки и в новых версиях pandas от него уходят.

In [ ]:
# Работаем с копией, чтобы не портить исходную таблицу
trips = df.copy()

# Вычисляемые признаки
trips["cost_per_km"] = (trips["cost_rub"] / trips["distance_km"]).round(2)
trips["speed_kmh"] = (trips["distance_km"] / trips["duration_h"]).round(1)
trips["is_delayed"] = trips["delay_h"] > 0            # сразу bool

print(trips[["trip_id", "distance_km", "cost_rub", "cost_per_km", "speed_kmh", "is_delayed"]].head())
print()
print("доля рейсов с задержкой: {:.1%}".format(trips["is_delayed"].mean()))

In [ ]:
# Категоризация числового столбца
trips["distance_group"] = pd.cut(
    trips["distance_km"],
    bins=[0, 300, 600, 1000],
    labels=["короткий", "средний", "длинный"],
)
print(trips["distance_group"].value_counts())
print()

# Переименование и удаление
trips = trips.rename(columns={"cost_rub": "cost"})
print("столбцы после rename:", list(trips.columns))

# Сортировка: самые дорогие километры
print()
print(trips.sort_values("cost_per_km", ascending=False)
          .head(5)[["trip_id", "route", "cargo_type", "cost_per_km"]])

### Задание 5

Создайте в таблице `trips` новые признаки:

1. `cost_per_ton` — стоимость за тонну груза (`cost` / `weight_t`), округлить до целых.
2. `delay_group` — категория задержки: `нет` (0 часов), `малая` (1–3), `крупная` (4 и больше).
   Подсказка: `pd.cut` с `bins=[-1, 0, 3, 100]`. Подумайте, почему левая граница `-1`, а не `0`.
3. `weight_filled` — вес груза, где пропуски заменены средним весом по таблице
   (метод `.fillna(...)`).

Затем выведите 10 рейсов с самой высокой стоимостью за тонну — только столбцы
`trip_id`, `route`, `weight_t`, `cost_per_ton`.

In [ ]:
# 1)
trips["cost_per_ton"] = ...

# 2)
trips["delay_group"] = ...

# 3)
trips["weight_filled"] = ...

# вывод топ-10
...

---
# Часть 6. Применение функций к строкам и столбцам

Три инструмента, и важно не путать, когда какой нужен:

| Инструмент | Применяется к | Когда использовать |
|---|---|---|
| `.map(словарь)` | значениям одного столбца | подстановка по справочнику |
| `.apply(функция)` | столбцу целиком | своя логика для одного столбца |
| `.apply(функция, axis=1)` | **каждой строке** | логика, которой нужно несколько столбцов |
| `.agg([...])` | столбцам | несколько статистик сразу |

**Важное предупреждение.** `.apply(axis=1)` — самый медленный способ в pandas: он действительно
проходит строки по одной, как обычный цикл. Если то же можно записать арифметикой по столбцам
(векторизованно) — записывайте арифметикой. Ниже мы измерим разницу.

In [ ]:
# map по словарю: подставляем тариф по типу груза
tariff_map = {
    "Контейнер": 7.5, "Нефтепродукты": 9.2, "Зерно": 5.1,
    "Металл": 6.8, "Лес": 4.6, "Стройматериалы": 5.9,
}
trips["tariff"] = trips["cargo_type"].map(tariff_map)
print(trips[["cargo_type", "tariff"]].head())
print()

# apply к одному столбцу: своя функция
def carrier_code(name):
    # код перевозчика: первые четыре буквы заглавными
    return name[:4].upper()

trips["carrier_code"] = trips["carrier"].apply(carrier_code)
print(trips[["carrier", "carrier_code"]].drop_duplicates())

In [ ]:
# apply(axis=1): функция получает всю строку и может использовать любые столбцы
def penalty(row):
    # Штраф за задержку: для нефтепродуктов 2000 руб/час, для остального 800
    if row["delay_h"] == 0:
        return 0
    rate = 2000 if row["cargo_type"] == "Нефтепродукты" else 800
    return row["delay_h"] * rate

trips["penalty"] = trips.apply(penalty, axis=1)
print(trips[["trip_id", "cargo_type", "delay_h", "penalty"]].head(8))
print()
print("всего штрафов:", trips["penalty"].sum(), "руб.")

In [ ]:
# Несколько статистик сразу по нескольким столбцам
trips[["weight_t", "distance_km", "cost", "cost_per_km"]].agg(["mean", "min", "max"]).round(1)

In [ ]:
# Векторизация против apply: то же самое, но по-разному
import time

def bench(fn, n=20):
    start = time.perf_counter()
    for _ in range(n):
        fn()
    return (time.perf_counter() - start) / n * 1000

# вариант 1: apply по строкам
apply_ms = bench(lambda: trips.apply(lambda r: r["cost"] / r["distance_km"], axis=1))

# вариант 2: арифметика по столбцам
vector_ms = bench(lambda: trips["cost"] / trips["distance_km"])

print(f"apply(axis=1): {apply_ms:.2f} мс")
print(f"векторизация:  {vector_ms:.2f} мс")
print(f"разница: {apply_ms / vector_ms:.0f} раз")

### Задание 6

1. Через `.map()` подставьте рейтинг перевозчика в новый столбец `carrier_rating`, используя
   словарь `ratings` ниже.
2. Напишите функцию `grade(row)`, которая по строке возвращает оценку рейса:
   - `"отлично"` — задержки нет и скорость больше 55 км/ч;
   - `"нормально"` — задержка не больше 2 часов;
   - `"плохо"` — во всех остальных случаях.

   Примените её через `.apply(..., axis=1)` в столбец `grade` и выведите `value_counts()`.
3. Посчитайте через `.agg()` среднее и максимум для столбцов `cost_per_km` и `speed_kmh`.
4. Тот же расчёт, что в пункте 2, невозможно записать одной арифметической операцией — почему?
   А вот `cost_per_km` можно. Запишите словами, в чём разница между этими двумя случаями.

In [ ]:
ratings = {
    "ТрансЛогистик": 4.6, "РусКарго": 4.1, "СибПуть": 3.8,
    "ВолгаТранс": 4.4, "АвтоДорСервис": 3.5,
}

# 1)
trips["carrier_rating"] = ...

# 2)
def grade(row):
    ...

trips["grade"] = ...
print(...)

# 3)
...

> **Ответ на пункт 4:**
>
> ...

---
# Часть 7. Мини-проект: карточка маршрута

Руководитель просит «карточку» по маршруту **Москва-Казань**: короткую сводку, которую можно
положить на одну страницу. Соберите её из того, что изучили.

**Что должно быть в карточке:**

1. число рейсов по маршруту;
2. суммарная выручка;
3. средняя стоимость за километр;
4. средний вес груза (пропуски в расчёт не берём);
5. доля рейсов с задержкой, в процентах;
6. самый частый тип груза;
7. таблица из 5 самых дорогих рейсов маршрута: `trip_id`, `cargo_type`, `cost`, `cost_per_km`.

Выводите аккуратно, с заголовком и выравниванием — как в отчёте, а не как отладочную печать.

In [ ]:
route_name = "Москва-Казань"
rt = trips[trips["route"] == route_name].copy()

print(f"КАРТОЧКА МАРШРУТА: {route_name}")
print("-" * 44)

# ваш код

---
## Бонус: взгляд вперёд

Карточку вы собрали по одному маршруту. А если нужно сразу по всем восьми? Переписывать фильтр
восемь раз не придётся — для этого есть `groupby`, и это тема следующей пары. Посмотрите,
как это выглядит: одна строка вместо восьми повторов.

In [ ]:
summary = trips.groupby("route").agg(
    trips_count=("trip_id", "count"),
    revenue=("cost", "sum"),
    avg_cost_per_km=("cost_per_km", "mean"),
    delayed_share=("is_delayed", "mean"),
).round(2).sort_values("revenue", ascending=False)

summary

---
## Отправляем работу

```
# терминал
git add .
git commit -m "Практика 6: Series и DataFrame, карточка маршрута"
git push
```

## Чек-лист сдачи

- [ ] Задание 1: обращение к `Series` по метке и по позиции, расчёт стоимости
- [ ] Задание 2: пять ответов о датасете с числами
- [ ] Задание 3: выборки через `loc` и `iloc`, рейс `T0250` найден по индексу
- [ ] Задание 4: пять фильтров, в каждом правильно расставлены скобки и `&` / `|`
- [ ] Задание 5: три новых признака, топ-10 по стоимости за тонну
- [ ] Задание 6: `map`, `apply(axis=1)`, `agg` и письменный ответ на пункт 4
- [ ] Задание 7: карточка маршрута оформлена
- [ ] Всё запушено в репозиторий

## Что унести с пары

Главное различие дня — между **метками и позициями** (`loc` против `iloc`) и между
**векторизацией и построчной обработкой** (арифметика по столбцам против `apply(axis=1)`).
Первое определяет, получите ли вы те строки, которые имели в виду. Второе — будет ли ваш
расчёт идти секунду или двадцать минут, когда строк станет миллион.